In [1]:
import pandas as pd
import numpy as np

quiz_df = pd.read_csv(
    "../data/analytics/quiz_performance.csv",
    parse_dates=["date"]
)

quiz_df.head()

,student_id,quiz_id,topic,score,max_score,date
0,S0001,QZ_LIN,Linear Algebra,62,100,2025-07-27 13:55:37.168988
1,S0001,QZ_LIN,Linear Algebra,83,100,2025-10-10 13:55:37.168988
2,S0001,QZ_LIN,Linear Algebra,72,100,2025-09-20 13:55:37.168988
3,S0001,QZ_LIN,Linear Algebra,44,100,2025-09-21 13:55:37.168988
4,S0001,QZ_PRO,Probability,34,100,2025-10-21 13:55:37.168988


In [2]:
student_quiz_stats = (
    quiz_df
    .groupby("student_id")
    .agg(
        avg_score=("score", "mean"),
        score_std=("score", "std"),
        attempts=("score", "count")
    )
    .reset_index()
)

student_quiz_stats.head()

,student_id,avg_score,score_std,attempts
0,S0001,64.347826,20.737776,23
1,S0002,63.086957,20.747208,23
2,S0003,63.615385,22.085429,26
3,S0004,63.083333,20.259011,24
4,S0005,53.916667,23.881804,24


In [3]:
high_score_thresh = student_quiz_stats["avg_score"].quantile(0.95)
low_std_thresh = student_quiz_stats["score_std"].quantile(0.10)

student_quiz_stats["anomaly_flag"] = (
    (student_quiz_stats["avg_score"] >= high_score_thresh) &
    (student_quiz_stats["score_std"] <= low_std_thresh) &
    (student_quiz_stats["attempts"] >= 3)
).astype(int)

student_quiz_stats["anomaly_flag"].value_counts()

anomaly_flag
0    492
1      8
Name: count, dtype: int64

In [4]:
anomalies = student_quiz_stats[
    student_quiz_stats["anomaly_flag"] == 1
]

anomalies.head()

,student_id,avg_score,score_std,attempts,anomaly_flag
37,S0038,73.480000,15.921997,25,1
99,S0100,74.538462,17.738615,26,1
184,S0185,70.791667,18.194550,24,1
310,S0311,74.666667,18.008855,24,1
322,S0323,70.400000,16.340135,25,1


In [5]:
student_quiz_stats.to_csv(
    "../data/analytics/academic_integrity_flags.csv",
    index=False
)

print("✅ Academic integrity analysis saved")

✅ Academic integrity analysis saved
